#TAREA 1

- Instalar en un entorno local o ejecutar Spark en algún servidor en línea (como Google Colab)

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
#Lo termine instalando en google colab para fines practicos pero adjunto el link del video de instalacion que use en mi windows 11
#https://www.youtube.com/watch?v=11srr2u2JRE

In [14]:
spark

- Elegir un conjunto de datos para trabajar durante el tetramestre, definirlo y explicar por qué se elige

###Conjunto de datos a trabajar durante el tetramestre:
Hice un dataset mediante la API de Youtube, donde me traje los comentarios de los ultimos 20-30 videos de los siguientes canales de habla hispana:
- Luisito Comunica
- Conversaciones (de Bandido Diamante y Daniel Migraña)
- El Rincon de Giorgio (de Jordi Wild)
- Mi Querido Mussolini (de Drossrotzank)

###¿Cual es el proposito de elegir semejante mezcla tan extraña de canales con audiencias tan distintas?
El proposito principal es mezclar generos de videos, tanto humor, como tematica de cultura y viajes, como tematicas mas serias o fuertes, asi como temas de actualidad y reflexion. Con el fin de poder estudiar o entender el tipo de publico que juntan en las cajas de comentarios, asi como las posibles motivaciones de las comunidades.

Nunca faltan los comentarios donde revelan su nacionalidad, sus emociones de disgusto o fanatismo, por lo cual es una tematica interesante o la denominaria "Off The Wall".

Como un preview, para esta tarea solo genere el dataset de 90 mil comentarios (filas) con 9 columnas, principalmente para no sobrecargar las consultas a la API, podria extender un poco mas el dataset no solo a mas numero de comentarios, si no a un mayor numero de canales que compongan el dataset.





- Cargar el conjunto de datos mediante PySpark (No hace falta cargarlo en el repositorio)

In [6]:
#Cargamos el dataset en colab y guardamos la ruta para leerlo
DATA_DIR="/content/youtube_dataset_comment.parquet"

In [13]:
#cargamos e imprimimos un ejemplo de la data recolectada
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

t0 = time.perf_counter()
spark = (
    SparkSession.builder
    .appName("Ejemplo del dataset de comentarios de Youtube")
    .getOrCreate()
)

df_s = spark.read.parquet(DATA_DIR)

top_spark = df_s.limit(10).toPandas()

t1 = time.perf_counter()

#Mostramos aca bonito como si fuese pandas
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.precision", 3)

display(top_spark)
print(f"PySpark tiempo: {t1 - t0:.3f} s")

,channel_id,channel_title,video_id,video_title,author,comment,likes,published_at,reply_count
0,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,IffJxvWNUiM,Contactar a Leo Messi fácilmente,@omarvalenteplazachapa,mañana nuevo capitulo de conversaciones,0,2026-03-08T22:32:07Z,0
1,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@jeffersonsilva1089,Soy de Nicaragua y cuando el dijo 1:...,0,2026-03-08T19:13:35Z,0
2,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@JorgeDeLara-xg3uw,Adrian solo fue de metiche y de mama...,0,2026-03-08T16:09:53Z,0
3,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@miguelangelherrera5732,hahahagagagha....c le maman a ese va...,0,2026-03-07T15:10:16Z,0
4,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@Armando-hw7bs,Demasiada cromadera para un tipo que...,1,2026-03-06T16:13:47Z,0
5,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@MauroVG-b3w,"Una reflexion al compa gonza, caga i...",0,2026-03-05T21:01:45Z,0
6,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@1pepmart,De donde es Gonzalo?,0,2026-03-05T20:39:54Z,0
7,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@1pepmart,De donde es Gonzalo?,0,2026-03-05T20:39:23Z,0
8,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@Noessuerte1,Cromadero full,0,2026-03-05T14:39:32Z,0
9,UCRZpxmNB22q_jXcLlk5N8Kw,Conversaciones,cxrvVgQ4bt4,Así se LOGRÓ el PODCAST con MESSI | ...,@oqart,Platicando con unos amigos argentino...,0,2026-03-04T23:21:00Z,0


PySpark tiempo: 0.419 s


In [16]:
df_s.printSchema() #vemos las columnas como en clase

root
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- video_title: string (nullable = true)
 |-- author: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- likes: long (nullable = true)
 |-- published_at: string (nullable = true)
 |-- reply_count: long (nullable = true)



- Usar PySpark para filtrar datos, generar estadísticas descriptivas básicas y realizar algunas operaciones aritméticas entre registros y columnas

In [44]:
#Filtramos datos
print("|||||||||||||| Solo comentarios con mas de 20,000 likes |||||")
df_s.filter(df_s.likes > 20000).show()

print("|||||Ahora solo comentarios del canal de Conversaciones con mas de 30 likes pero ordenados del mayor numero de respuestas al menor |||||")
df_s.filter((df_s.likes > 30) & (df_s.channel_title == "Conversaciones") ).orderBy(F.desc("reply_count")).show()
#use 30 porque suele ser un canal con pocos comentarios por video

print("|||| Comentarios del canal 'Mi Querido Mussolini' con más de 10 likes y menos de 5 caracteres de largo||||")
df_s.filter((df_s.likes > 10) & (df_s.channel_title == "Mi Querido Mussolini") &(F.length(df_s.comment) < 5)).show()

print("|||||Los primeros 5 comentarios del canal de Luisito Comunica que mencionen a Cuba con mayor numero de likes||||")
df_s.filter((df_s.channel_title == "Luisito Comunica") & (df_s.comment.contains("cuba")) ).orderBy(F.desc("likes")).show(5)
#Esto porque los comentarios en el canal de luisito son un portal a miles de partes del mundo y siempre hay gente de otros paises contando sus vivencias

|||||||||||||| Solo comentarios con mas de 20,000 likes |||||
+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+
|          channel_id|       channel_title|   video_id|         video_title|              author|             comment|likes|        published_at|reply_count|
+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+-----+--------------------+-----------+
|UCDZsyOkn-WTiTwgA...|El Rincón De Giorgio|5slQdCUb_IM|EL YOUTUBER QUE S...|       @futjuako1883|Ver a Giorgio con...|20542|2025-10-31T17:07:17Z|        118|
|UCDZsyOkn-WTiTwgA...|El Rincón De Giorgio|5slQdCUb_IM|EL YOUTUBER QUE S...|  @myheartverybroken|SE ACORDÓ DE LA C...|37685|2025-10-31T15:32:08Z|        205|
|UCDZsyOkn-WTiTwgA...|El Rincón De Giorgio|ZkKSvb73NHo|LA HISTORIA REAL ...|         @FractorX12|SE ACORDÓ DE LA C...|45738|2025-10-01T16:30:57Z|   

In [17]:
#estadisticas descriptivas
df_s.describe().show()

+-------+--------------------+--------------------+-----------+--------------------+-----------+--------------------+------------------+--------------------+-------------------+
|summary|          channel_id|       channel_title|   video_id|         video_title|     author|             comment|             likes|        published_at|        reply_count|
+-------+--------------------+--------------------+-----------+--------------------+-----------+--------------------+------------------+--------------------+-------------------+
|  count|               90000|               90000|      90000|               90000|      90000|               89960|             90000|               90000|              90000|
|   mean|                NULL|                NULL|       NULL|                NULL|       NULL|  375.05227272727274|14.662788888888889|                NULL|0.21096666666666666|
| stddev|                NULL|                NULL|       NULL|                NULL|       NULL|   794.4593328

In [62]:
#operaciones artimeticas
print("|||||Rankeamos el total de interacciones en el canal de Luisito Comunica |||||")#sumamos 2 columnas y ya xd
display(df_s.filter(df_s.channel_title == "Luisito Comunica").withColumn("interacciones",df_s.likes + df_s.reply_count).select("author","comment", "likes", "reply_count", "interacciones").orderBy(F.desc("interacciones")).limit(10).toPandas()
)#mejor imprimimos los primeros 10

print("------------------------")
print("|||||Canales pero por numero de comentarios |||||")
df_s.groupBy("channel_title").count().orderBy(F.desc("count")).toPandas()#hacemos un select distinct pero en pyspark en pocas palabras y agrupado por canales


|||||Rankeamos el total de interacciones en el canal de Luisito Comunica |||||


,author,comment,likes,reply_count,interacciones
0,@cinthiaarvizu9191,Trabaje en un lugar donde vendía per...,10892,140,11032
1,@roddsegovia,Todos somos liberales hasta que toca...,10324,114,10438
2,@karazu121,Editado pq ya me aburri de las respu...,7570,352,7922
3,@mamitaoreilly7405,El problema es que ellos no se adapt...,7266,118,7384
4,@borisalbertocastilloleiva7464,Definitivamente la diferencia no est...,6967,121,7088
5,@Ltrrs,Tienen que adaptarse a donde vayan. ...,4908,162,5070
6,@jhonyxxx159,Soy de Perú y me da vergüenza ajena ...,2817,232,3049
7,@ivonneykata,Trabajo en una fábrica en Toronto y ...,2368,60,2428
8,@crisis5629,Es porque no es solo una app de “bai...,1952,28,1980
9,@DiegoFerdynan,siento que cada ciudad del mundo tie...,1936,31,1967


------------------------
|||||Canales pero por numero de comentarios |||||


,channel_title,count
0,El Rincón De Giorgio,71250
1,Luisito Comunica,13709
2,Mi Querido Mussolini,4887
3,Conversaciones,154
